# kin_sandbox — getting fluent with the kinematics core (library 0.5.0)

A playground for the new pipeline, run on sessions it has already processed. Nothing here is
analysis to keep; change anything, rerun, break it.

**Examples:**
1. What a session folder holds
2. One lick, frame by frame
3. Trials: what the task module already summarizes
4. Your own per-trial summary (`summarize_by_window`)
5. Movements over time: rates per bin and around the go cue
6. Licks just outside movements
7. Rerun the pipeline with your own settings (confidence threshold)
8. A body part that never disappears: the jaw
9. Add your own measurement (`add_feature_group`)
10. Pool sessions

Design, column map and verification: `KINEMATICS_PLAN.md` in the library. Every module has a
docstring: `help(pl)`, `help(segment)`, ...

## Setup

In [ ]:
%matplotlib inline
import dataclasses
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotstyle import apply_style, OKABE_ITO, PALETTE, style_ax
apply_style()
OUT, BACK = OKABE_ITO["blue"], OKABE_ITO["vermillion"]   # protrusion, retraction

In [ ]:
# The library: version 0.5.0. Until the capsule image is rebuilt with it, use a checkout of
# the v0.5.0 tag (the one made for the comparison).
LIB_CHECKOUT = Path("/root/capsule/scratch/kinematics_compare/dfbva_v050/src")
try:
    from aind_dynamic_foraging_behavior_video_analysis.kinematics.tasks import (
        dynamic_foraging_tongue as dft,
    )
except ImportError:
    sys.path.insert(0, str(LIB_CHECKOUT))
    for name in [m for m in sys.modules if m.startswith("aind_dynamic_foraging_behavior_video_analysis")]:
        del sys.modules[name]
    from aind_dynamic_foraging_behavior_video_analysis.kinematics.tasks import (
        dynamic_foraging_tongue as dft,
    )
import aind_dynamic_foraging_behavior_video_analysis as lib
from aind_dynamic_foraging_behavior_video_analysis.kinematics import (
    features, intervals, landmarks, pipeline as pl, segment, signal,
)
print("library", lib.__version__, "from", Path(lib.__file__).parent)

In [ ]:
# Where processed sessions are: the full batch if it has run, else the five test sessions.
if "KIN_SANDBOX_ROOT" in os.environ:                       # for running this notebook elsewhere
    candidates = [Path(os.environ["KIN_SANDBOX_ROOT"])]
else:
    candidates = [Path("/root/capsule/scratch/kinematics_v050"),
                  Path("/root/capsule/scratch/kinematics_compare/new_r4")]
OUT_ROOT = next(p for p in candidates if p.exists() and any(p.glob("*/record.json")))
SESSIONS = sorted(p.parent.name for p in OUT_ROOT.glob("*/record.json"))
SESSION_DIR = OUT_ROOT / SESSIONS[0]                       # change the index to switch session
TONGUE, JAW = "tongue_tip_center", "jaw"
print(OUT_ROOT, f"{len(SESSIONS)} sessions; using", SESSION_DIR.name)

## 1. What a session folder holds

One parquet file per table and `record.json`. `pl.read` reads them; ask only for the tables you
need, since `pose` and `kinematics` are one row per frame per keypoint.

In [ ]:
for p in sorted(SESSION_DIR.iterdir()):
    print(f"{p.name:<22} {p.stat().st_size / 1e6:8.1f} MB")
d = pl.read(SESSION_DIR, tables=["movements", "submovements", "bouts", "trials", "licks", "landmarks"])
for name, t in d.items():
    if isinstance(t, pd.DataFrame):
        print(f"\n{name}: {t.shape}\n  {list(t.columns)}")

In [ ]:
rec = d["record"]
print("clock:", rec["clock"])
print("frames:", rec["frames"])
print("timing:", {k: rec["timing"][k] for k in ("verdict", "harp_sources")})
print("quality:", json.dumps(rec["quality"], indent=1))
print("versions:", rec["versions"])
# The settings that produced this folder, as a Settings object you can change and rerun:
settings = pl.Settings.from_dict(rec["settings"])
print(settings.keypoints[0].segments)

## 2. One lick, frame by frame

A movement with a lick that splits into one protrusion and one retraction. Frames carry the ids
of every segment they belong to, so a movement's frames are one filter away.

In [ ]:
mv, sub, licks, lm = d["movements"], d["submovements"], d["licks"], d["landmarks"]
jaw_xy = landmarks.origin(lm, JAW)
n_pieces = sub.groupby("movement_id").size()
with_lick = mv[mv.n_licks == 1]
candidates = with_lick[with_lick.movement_id.isin(n_pieces[n_pieces == 2].index)]
if candidates.empty:                                       # fall back to any split lick
    candidates = with_lick[with_lick.movement_id.isin(n_pieces[n_pieces >= 2].index)]
m = candidates.iloc[len(candidates) // 2]                  # a typical one; try others
kin = pd.read_parquet(SESSION_DIR / "kinematics.parquet", filters=[("keypoint", "==", TONGUE)])
f = kin[kin.movement_id == m.movement_id]
t_ms = (f.time - m.start_time) * 1000
direction = f.submovement_id.map(sub.set_index("submovement_id").direction)

fig, ax = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
ax[0].plot(t_ms, np.hypot(f.x - jaw_xy[0], f.y - jaw_xy[1]), color=PALETTE["all"])
ax[0].set_ylabel("distance from jaw (px)")
ax[1].bar(t_ms, f.radial_velocity, width=1.6, color=np.where(direction > 0, OUT, BACK))
ax[1].axhline(0, color=PALETTE["neutral"], lw=0.8)
ax[1].set_ylabel("radial velocity (px/s)")
ax[1].set_xlabel("time from movement start (ms)")
for t in licks.loc[licks.movement_id == m.movement_id, "timestamps"]:
    for a in ax:
        a.axvline((t - m.start_time) * 1000, color=PALETTE["accent"], ls="--", lw=1)
ax[0].set_title(f"movement {m.movement_id}: amplitude {m.amplitude:.0f} px, "
                f"angle {m.angle_deg:.0f}°, lick (dashed) at {(m.lick_time - m.start_time) * 1000:.0f} ms")
for a in ax:
    style_ax(a)
print(sub[sub.movement_id == m.movement_id][["submovement_id", "number_in_movement", "direction",
                                             "n_frames", "radial_velocity_sum", "amplitude"]])

## 3. Trials: what the task module already summarizes

`trials` is the NWB trials table plus movement and lick summaries. A trial runs from its go cue
to the next one.

In [ ]:
tr = d["trials"]
cols = ["trial", "goCue_start_time_in_session", "n_movements", "n_lick_movements",
        "first_movement_latency", "cue_response_movement_number", "lick_latency", "lick_coverage"]
print(tr[[c for c in cols if c in tr]].head(8).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(tr.first_movement_latency.dropna(), bins=np.arange(0, 2, 0.02), color=PALETTE["all"])
ax[0].set_xlabel("first movement after the go cue (s)")
ax[1].scatter(tr.first_movement_latency, tr.lick_latency, s=6, alpha=0.4, color=PALETTE["all"])
ax[1].plot([0, 2], [0, 2], color=PALETTE["neutral"], lw=0.8)
ax[1].set_xlim(0, 2); ax[1].set_ylim(0, 2)
ax[1].set_xlabel("first movement (s)"); ax[1].set_ylabel("first cue-response lick (s)")
for a in ax:
    style_ax(a)

## 4. Your own per-trial summary

`intervals.summarize_by_window(items, windows, time_col, how)` puts each item in the window its
time falls in, then summarizes per window. Windows can be anything: here, the first 2 s after
each go cue. `how` maps a new column to `(item column, summary)`; the summary is `'count'`,
`'first'` / `'last'` (in time order), a pandas name (`'mean'`, `'median'`, ...) or a function.

In [ ]:
go = tr.sort_values("goCue_start_time_in_session")
win = pd.DataFrame({"trial": go.trial.to_numpy(),
                    "start_time": go.goCue_start_time_in_session.to_numpy()})
nxt = np.r_[win.start_time.to_numpy()[1:], np.inf]
win["end_time"] = np.minimum(win.start_time + 2.0, nxt)    # windows must not overlap

early = intervals.summarize_by_window(mv, win, "start_time", {
    "n_movements_2s": (None, "count"),
    "n_lick_movements_2s": ("n_licks", lambda s: int((s > 0).sum())),
    "mean_amplitude_2s": ("amplitude", "mean"),
    "median_peak_speed_2s": ("peak_speed", "median"),
    "first_angle_2s": ("angle_deg", "first"),
})
print(early.head(8).to_string(index=False))

## 5. Movements over time

Fixed time bins are windows too (`intervals.time_bins`): movement rate across the session shows
engagement. For a go-cue-aligned histogram, plain numpy is simplest.

In [ ]:
bins = intervals.time_bins(0, mv.end_time.max(), 10.0)
rate = intervals.summarize_by_window(mv, bins, "start_time", {"n": (None, "count")})

go_t = go.goCue_start_time_in_session.to_numpy()
rel = mv.start_time.to_numpy()[:, None] - go_t[None, :]
rel = rel[(rel >= -1) & (rel < 3)]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(rate.start_time / 60, rate.n / 10, color=PALETTE["all"])
ax[0].set_xlabel("session time (min)"); ax[0].set_ylabel("movements / s (10 s bins)")
ax[1].hist(rel, bins=np.arange(-1, 3, 0.02), color=PALETTE["all"])
ax[1].axvline(0, color=PALETTE["neutral"], ls="--")
ax[1].set_xlabel("movement start relative to go cue (s)"); ax[1].set_ylabel("movements")
for a in ax:
    style_ax(a)

## 6. Licks just outside movements

A lick is matched to a movement only if it falls inside it (within half a frame). Every lick also
records the closest movement and the signed time to it, so you can see near misses and apply a
looser rule yourself.

In [ ]:
off_ms = licks.closest_movement_offset_s * 1000
print(f"inside a movement: {licks.movement_id.notna().mean():.3f}   "
      f"within 10 ms: {(off_ms.abs() <= 10).mean():.3f}   (record: {rec['quality']['lick_coverage']:.3f})")
fig, ax = plt.subplots(figsize=(7, 3))
outside = off_ms[off_ms != 0]
ax.hist(outside.clip(-30, 30), bins=np.arange(-30, 31, 1), color=PALETTE["all"])
ax.set_xlabel("lick time minus closest movement (ms); <0 before its start, >0 after its end")
ax.set_ylabel("licks outside a movement")
style_ax(ax)

# A looser rule, without rerunning anything:
loose_id = licks.closest_movement_id.where(off_ms.abs() <= 5)
print("coverage with a 5 ms rule:", round(loose_id.notna().mean(), 3))

## 7. Rerun the pipeline with your own settings

The saved `pose` table already has time, so `pl.process` reruns everything without touching the
raw files. Here: the first 10 minutes, tongue movements at two confidence thresholds. A lower
threshold starts movements earlier (closer to the jaw) and can rescue licks the tracker saw late,
at the risk of accepting wrong detections. Settings objects are frozen; `dataclasses.replace`
makes changed copies.

In [ ]:
pose = pd.read_parquet(SESSION_DIR / "pose.parquet")
pose_10 = pose[(pose.time >= 0) & (pose.time < 600)]
licks_10 = licks[(licks.timestamps >= 0) & (licks.timestamps < 600)][["timestamps", "cue_response", "rewarded", "event"]]
tongue = dft.TONGUE_SETTINGS.keypoints[0]

rows = []
for thr in (0.9, 0.7, 0.5):
    kp = dataclasses.replace(tongue, min_likelihood=thr, segments={"movement": tongue.segments["movement"]})
    s = dataclasses.replace(dft.TONGUE_SETTINGS, keypoints=(kp,))
    res = pl.process(pose_10, s)
    m10, l10 = dft.match_licks(res["movements"], licks_10, "movement_id")
    rows.append({"min_likelihood": thr, "movements": len(m10),
                 "start distance from jaw (px)": np.hypot(m10.start_x, m10.start_y).median(),
                 "duration (ms)": 1000 * m10.duration.median(),
                 "lick coverage": l10.movement_id.notna().mean()})
pd.DataFrame(rows).round(3)

## 8. A body part that never disappears: the jaw

The jaw is always tracked, so "runs of tracked frames" would be one segment. Instead segment where
it moves: `above_threshold` on speed, with two levels so noise near one level does not chatter,
gaps between licks bridged so a licking bout is one segment. First look at the speeds to pick the
levels, then measure each segment's dominant frequency along the axis the jaw moves most.

In [ ]:
jaw_only = pl.Settings(fs_hz=500.0, keypoints=(pl.KeypointSettings(JAW, smooth=pl.Smooth(cutoff_hz=30.0)),))
jk = pl.process(pose_10, jaw_only)["kinematics"]
q = jk.speed.quantile([0.5, 0.75, 0.9, 0.99])
axis = "x" if jk.x.std() > jk.y.std() else "y"
print("jaw speed quantiles (px/s):", q.round(1).to_dict(), "  moves most along", axis)

jaw_kp = pl.KeypointSettings(
    JAW, smooth=pl.Smooth(cutoff_hz=30.0),
    segments={"jaw_movement": pl.Segmentation(
        "above_threshold",
        {"column": "speed", "on": float(q[0.9]), "off": float(q[0.5]), "max_gap_frames": 50, "min_frames": 250},
        features=("path", "spectrum"))},
    feature_options={"spectrum": {"column": axis, "band": (2.0, 20.0), "min_duration_s": 0.5}},
)
jres = pl.process(pose_10, dataclasses.replace(jaw_only, keypoints=(jaw_kp,)))
jm = jres.get("jaw_movements", pd.DataFrame())
print(len(jm), "jaw segments of at least 0.5 s")
bouts_10 = d["bouts"][d["bouts"].start_time < 600]

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(jm.get("dominant_hz", pd.Series(dtype=float)).dropna(), bins=np.arange(2, 20, 0.5),
        alpha=0.7, color=OKABE_ITO["bluish_green"], label="jaw: dominant frequency")
ax.hist(bouts_10.rate_hz.dropna(), bins=np.arange(2, 20, 0.5), alpha=0.7,
        color=OKABE_ITO["blue"], label="tongue: movements per s within a bout")
ax.set_xlabel("Hz"); ax.legend(frameon=False)
style_ax(ax)
plt.show()

## 9. Add your own measurement

A feature group is a function of one segment's frames (`seg`: dict of arrays; positions are
smoothed pixels, NaN where not valid) and a context, returning a dict of columns. Register it with
`features.add_feature_group` and list it in a segmentation's `features`. Here: how close each
movement gets to each spout, which should be near zero for movements with a lick.

In [ ]:
spouts = lm.set_index("keypoint").reindex(["spout_l", "spout_r"])[["x", "y"]].dropna()

@features.add_feature_group("spout_reach")
def spout_reach(seg, context):
    v = seg["valid"]
    x, y = seg["x"][v], seg["y"][v]
    return {f"closest_to_{name}_px": float(np.min(np.hypot(x - sx, y - sy))) if len(x) else np.nan
            for name, (sx, sy) in spouts.iterrows()}

kp = dataclasses.replace(tongue, segments={"movement": dataclasses.replace(
    tongue.segments["movement"], features=("path", "excursion", "spout_reach"))})
res = pl.process(pose_10, dataclasses.replace(dft.TONGUE_SETTINGS, keypoints=(kp,)))
m10, _ = dft.match_licks(res["movements"], licks_10, "movement_id")
reach_cols = [c for c in m10 if c.startswith("closest_to_")]
m10["closest_spout_px"] = m10[reach_cols].min(axis=1)
print(m10.groupby(m10.n_licks > 0)["closest_spout_px"].describe().round(1))

fig, ax = plt.subplots(figsize=(7, 3))
for has, color, label in ((True, OKABE_ITO["vermillion"], "with a lick"), (False, PALETTE["not_sig"], "no lick")):
    ax.hist(m10.loc[(m10.n_licks > 0) == has, "closest_spout_px"], bins=np.arange(0, 80, 2),
            alpha=0.7, color=color, label=label)
ax.set_xlabel("closest approach to a spout (px)"); ax.legend(frameon=False)
style_ax(ax)
plt.show()

## 10. Pool sessions

Every table has `session`, and ids are unique within a session, so pooling is a concat and
grouping is by `session`. Read only the columns you need.

In [ ]:
per_session = []
for s in SESSIONS:
    sd = OUT_ROOT / s
    m_s = pd.read_parquet(sd / "movements.parquet",
                          columns=["session", "keypoint", "duration", "amplitude", "peak_speed", "n_licks"])
    m_s = m_s[m_s.keypoint == TONGUE]
    b_s = pd.read_parquet(sd / "bouts.parquet", columns=["rate_hz", "n_movements"])
    q_s = pl.read_record(sd)["quality"]
    per_session.append({"session": s, "movements": len(m_s),
                        "lick coverage": q_s["lick_coverage"],
                        "coverage within 10 ms": q_s["lick_coverage_within_10ms"],
                        "median amplitude (px)": m_s.amplitude.median(),
                        "median peak speed (px/s)": m_s.peak_speed.median(),
                        "median bout rate (Hz)": b_s.rate_hz.median()})
pd.DataFrame(per_session).round(3)

## Where next

- Change `SESSION_DIR` and rerun from the top.
- `help(segment.zero_crossings)`, `help(intervals.summarize_by_window)`, `help(pl.Segmentation)`
  for every parameter.
- Other keypoints in `pose` (`nose_tip`, paws, the other tongue points) can get their own
  `KeypointSettings` exactly like the jaw in example 8.